In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

In [2]:
batch_size = 2048
device = torch.device('cpu')

In [3]:
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('data', train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
    ])),
    batch_size=batch_size, shuffle=True)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('data', train=False, transform=transforms.Compose([
        transforms.ToTensor(),
    ])),
    batch_size=batch_size, shuffle=True)

print("训练数据形状:", train_loader.dataset.data.shape)
print("测试数据形式:", test_loader.dataset.data.shape)

训练数据形状: torch.Size([60000, 28, 28])
测试数据形式: torch.Size([10000, 28, 28])


In [13]:
class mlp(nn.Module):
    def __init__(self):
        super(mlp, self).__init__()
        # 第一层全连接：784输入 →128输出
        self.l1 = nn.Linear(784, 128)
        # 第二层全连接：128输入 →10输出（10个数字类别）
        self.l2 = nn.Linear(128, 10)

    def forward(self, x):
        a1 = self.l1(x)
        x1 = F.relu(a1)   # ReLU激活函数
        a2 = self.l2(x1)
        return a2

# 实例化模型并迁移到对应设备(GPU/CPU)
model = mlp().to(device)
# SGD优化器
optimizer = optim.SGD(model.parameters(), lr=0.01)
print(model)

mlp(
  (l1): Linear(in_features=784, out_features=128, bias=True)
  (l2): Linear(in_features=128, out_features=10, bias=True)
)


In [14]:
epochs = 30
for epoch in range(epochs):
    # ==========训练阶段==========
    model.train()
    for batch_idx, (x, y) in enumerate(train_loader):
        # 将28×28图片展平为784维向量，并搬运到设备
        x, y = x.view(x.shape[0], -1).to(device), y.to(device)
        optimizer.zero_grad()       # 梯度清零
        output = model(x)           # 前向传播
        loss = F.cross_entropy(output, y) # 计算交叉熵损失
        loss.backward()             # 反向传播求梯度
        optimizer.step()            # 更新参数

    # ==========测试评估阶段==========
    model.eval()
    correct = 0
    test_loss = 0
    with torch.no_grad(): # 关闭梯度计算
        for batch_idx, (x, y) in enumerate(test_loader):
            x, y = x.view(x.shape[0], -1).to(device), y.to(device)
            output = model(x)
            test_loss += F.cross_entropy(output, y)
            pred = output.max(1, keepdim=True)[1] # 获取预测类别
            correct += pred.eq(y.view_as(pred)).sum().item()

    test_loss = test_loss / (batch_idx+1)
    acc = correct / len(test_loader.dataset)
    print(f'epoch:{epoch:2d}, loss:{test_loss:.4f}, acc:{acc:.4f}')

epoch: 0, loss:2.2470, acc:0.3193
epoch: 1, loss:2.1892, acc:0.4825
epoch: 2, loss:2.1251, acc:0.5673
epoch: 3, loss:2.0539, acc:0.6117
epoch: 4, loss:1.9749, acc:0.6431
epoch: 5, loss:1.8884, acc:0.6689
epoch: 6, loss:1.7954, acc:0.6939
epoch: 7, loss:1.6974, acc:0.7089
epoch: 8, loss:1.5973, acc:0.7242
epoch: 9, loss:1.4976, acc:0.7382
epoch:10, loss:1.4021, acc:0.7518
epoch:11, loss:1.3114, acc:0.7651
epoch:12, loss:1.2282, acc:0.7777
epoch:13, loss:1.1526, acc:0.7880
epoch:14, loss:1.0844, acc:0.7974
epoch:15, loss:1.0241, acc:0.8073
epoch:16, loss:0.9698, acc:0.8149
epoch:17, loss:0.9219, acc:0.8204
epoch:18, loss:0.8786, acc:0.8254
epoch:19, loss:0.8408, acc:0.8290
epoch:20, loss:0.8064, acc:0.8334
epoch:21, loss:0.7752, acc:0.8383
epoch:22, loss:0.7473, acc:0.8423
epoch:23, loss:0.7225, acc:0.8459
epoch:24, loss:0.6996, acc:0.8488
epoch:25, loss:0.6784, acc:0.8525
epoch:26, loss:0.6595, acc:0.8552
epoch:27, loss:0.6423, acc:0.8585
epoch:28, loss:0.6262, acc:0.8601
epoch:29, loss